# Approach 2: full-data chunked Semantic IRT

This uses **all 229,143 eligible responses**, not Steven's 1,000-response sample. The cohort and recorded splits are exactly those from our latest MathArena-inclusive Approach 1 study. There are 571,921 downloaded rows: the existing correctness-prediction policy excludes MMDocRAG's 342,695 graded-credit rows and 83 fractional MathArena outcomes. MathArena's retained exact endpoints are a conditional population, not a claim that all its proof grades are intrinsically binary.

No response sampling or item cap is applied. The verified existing chunk cache supplies all 9,688 distinct prompt vectors, using the agreed 2,048-token content cap, 512-token windows and token-weighted normalized pooling. This avoids recomputing identical frozen embeddings.

The 4PL Semantic IRT architecture, row-weighted Brier/MSE objective and mean-ability fallback remain Steven's. Training settings remain the pilot's, so this is a new full-data run rather than a claim of a controlled architecture comparison.

**Item holdout:** use Approach 1's recorded outer item test (41,870 responses), separate internal validation (47,491) and training (139,782). All item/prompt groups stay disjoint. The outer item score is never used for checkpoint selection.

**Benchmark holdout:** run all five recorded benchmark tests. Each benchmark is absent from training and internal validation. All subjects remain included; unseen names use the mean training-subject ability fallback. No submission is uploaded, and the pilot artifacts remain unchanged.


In [ ]:
# Setup: run from this repository in the existing project Python environment.
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from holdout_training import (
    run_fold, score, learning_curves, comparison_plot,
)
from full_data_training import load_full_data, recorded_split, sha
from models.Semantic_IRT import Semantic_IRT
from models.text_chunking import TEXT_ENCODING

WORKSPACE = Path.cwd()
assert (WORKSPACE / "samples.json").is_file(), "Launch this notebook from the PSN-IRT directory."
torch.set_num_threads(2)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", sys.executable)
print("Training device:", DEVICE)
print("Torch:", torch.__version__)


## Full data and verified frozen embeddings

Raw source hashes must match the original study. The loader checks response identities, labels and subject names against the raw tables. It verifies the cached vectors against cache keys recomputed from current encoder files, libraries and matching chunking code. No Approach 1 source, dataset, split or cache is modified.


In [ ]:
APPROACH1_STUDY = WORKSPACE.parent / "my-approach1/experiments/approach1_matharena_binaryonly_20261020"
RAW_DATA = WORKSPACE.parent / "approach1-starter/data/raw"
LOCAL_ENCODER = WORKSPACE.parent / "approach1-starter/resources/text_encoder"
frame, item_vectors, data_audit = load_full_data(APPROACH1_STUDY, RAW_DATA, LOCAL_ENCODER)
display(pd.DataFrame(data_audit["inventory"]))
dataset_overview = frame.groupby("benchmark_id").agg(
    responses=("response", "size"), items=("item_id", "nunique"),
    subjects=("normalized_name", "nunique"), observed_success_rate=("response", "mean"),
)
display(dataset_overview)
print("Eligible responses:", len(frame), "| Distinct items:", frame.item_id.nunique(),
      "| Item/prompt groups:", frame.item_group.nunique())
print("Verified cached prompt vectors:", data_audit["unique_prompts"])
print("Encoding:", TEXT_ENCODING)


## Training settings and architecture

Keep the original 4PL architecture, AdamW learning rate 0.0003, weight decay 0.00001, 512-response batches and maximum 100 epochs. Restore the best equal subject-benchmark-pair **internal validation** Brier checkpoint; stop after 10 epochs without improvement. The learning-rate scheduler sees internal validation only.

Splits are loaded from the recorded Approach 1 membership; they are not resampled here. Choose a fresh run name to retrain. To revisit graphs, run Setup, the settings cell and the saved-results/plot cells while skipping data verification and training.


In [ ]:
RUN_NAME = "chunked_full_data_20261107_v1"
RUN_DIR = WORKSPACE / "results/chunked/experiments" / RUN_NAME
SETTINGS = {
    "seed": 5814, "epochs": 100, "batch_size": 512,
    "learning_rate": 3e-4, "weight_decay": 1e-5, "patience": 10,
}
print("4PL Semantic IRT | 384-dimensional item input | 64-unit hidden layers")
print("Run directory:", RUN_DIR)


## Fit the recorded item holdout and all five benchmark holdouts

The membership hash, original response order and every saved split index are checked before training. Every experiment accounts for all eligible responses, including any rows purged because of shared prompts. Only internal validation chooses the checkpoint. The model is unchanged; batched inference bounds intermediate memory for the larger response population.


In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=False)
source_names = ("holdout_training.py", "full_data_training.py", "models/Semantic_IRT.py", "models/text_chunking.py")
manifest = {
    "data_audit": data_audit, "approach1_study": str(APPROACH1_STUDY),
    "source_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "source_sha256": {name: sha(WORKSPACE / name) for name in source_names},
    "encoder_source": str(LOCAL_ENCODER),
    "encoder_provenance": json.loads((LOCAL_ENCODER / "source.json").read_text()),
    "text_encoding": TEXT_ENCODING, "settings": SETTINGS,
    "device": DEVICE, "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ("torch", "numpy", "pandas", "scikit-learn", "sentence-transformers", "transformers")},
    "rows": len(frame), "items": frame.item_id.nunique(),
    "evaluation": "Full eligible Approach 1 cohort and exact recorded grouped splits; independent outer item/benchmark tests; all subjects scored.",
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
for name in source_names:
    target = RUN_DIR / "source" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(WORKSPACE / name, target)
experiments = [("item_holdout", "item")] + [
    ("benchmark__" + benchmark, "benchmark__" + benchmark) for benchmark in sorted(frame.benchmark_id.unique())
]
comparisons = []
for name, recorded_name in experiments:
    split = recorded_split(frame, APPROACH1_STUDY, recorded_name)
    print("\n", name, {label: len(part) for label, part in split.items()})
    comparisons.append(run_fold(split, item_vectors, RUN_DIR / name, SETTINGS, name, DEVICE))
comparison = pd.concat(comparisons, ignore_index=True)
comparison.to_csv(RUN_DIR / "comparison.csv", index=False)
print("Saved all six full-data experiments:", RUN_DIR)


## Internal validation and independent item test

Internal validation selects the epoch; the outer item test is scored only after restoration. The baselines use constant 0.5 and the training-only global mean. Scores include every response, with the existing fallback for unseen subject names. Lower Brier is better.


In [ ]:
# Saved results: this cell and the plots work after restarting the kernel.
comparison = pd.read_csv(RUN_DIR / "comparison.csv")
columns = ["experiment", "predictor", "rows", "best_epoch", "row_Brier", "equal_pair_Brier", "BCE",
           "known_subject_rows", "unseen_subject_rows"]
validation_scores = comparison[comparison.partition == "validation"][columns]
display(validation_scores.round(4))

item_test_scores = comparison[(comparison.partition == "heldout") & (comparison.experiment == "item_holdout")][columns]
print("Independent item-held-out test:")
display(item_test_scores.round(4))


## Plot 1: learning curves / overfitting

Blue is training Brier; orange is validation Brier with equal response weighting. Green gives equal weight to each subject-benchmark pair and controls checkpoint selection. The vertical line marks the restored epoch. Diverging training and validation curves suggest overfitting.


In [ ]:
fig = learning_curves(RUN_DIR)
fig.savefig(RUN_DIR / "learning_curves.png", dpi=160, bbox_inches="tight")
plt.show()


## Plot 2: validation against constant baselines


In [ ]:
fig = comparison_plot(comparison, "validation")
fig.savefig(RUN_DIR / "validation_comparison.png", dpi=160, bbox_inches="tight")
plt.show()


## Benchmark holdout results

These rows contain only the outer benchmark, scored after checkpoint selection on other benchmarks. **Unseen subject rows remain included** using the mean-ability fallback. The known-only score is supplementary and must not replace the all-response denominator. A missing known-only value means that fold has no known subjects.


In [ ]:
benchmark_scores = comparison[(comparison.partition == "heldout") & (comparison.experiment != "item_holdout")][columns + ["known_only_row_Brier"]]
display(benchmark_scores.round(4))
model_holdouts = benchmark_scores[benchmark_scores.predictor == "Semantic IRT"]
print("Macro benchmark Brier (row metric, each benchmark equal):", model_holdouts.row_Brier.mean())
print("Macro benchmark Brier (equal-pair metric, each benchmark equal):", model_holdouts.equal_pair_Brier.mean())


In [ ]:
# The outer item and benchmark scores never select the checkpoint.
fig = comparison_plot(comparison, "heldout")
fig.savefig(RUN_DIR / "heldout_scores.png", dpi=160, bbox_inches="tight")
plt.show()


## Outer item test, broken down by benchmark

These are the independent held-out items from the single item experiment, after selecting a checkpoint on its separate internal validation set. They differ from holding an entire benchmark out of training.


In [ ]:
item_predictions = pd.read_csv(RUN_DIR / "item_holdout/predictions.csv")
item_test = item_predictions[item_predictions.partition == "heldout"]
rows = []
for benchmark, group in item_test.groupby("benchmark_id"):
    for predictor, column in (("Semantic IRT", "probability"), ("constant 0.5", "constant_0.5"),
                              ("training global mean", "training_global_mean")):
        rows.append({"experiment": benchmark, "partition": "heldout", "predictor": predictor,
                     "rows": len(group), **score(group, group[column])})
item_by_benchmark = pd.DataFrame(rows)
item_by_benchmark.to_csv(RUN_DIR / "item_test_by_benchmark.csv", index=False)
display(item_by_benchmark.round(4))
fig = comparison_plot(item_by_benchmark, "heldout")
fig.axes[0].set_title("Independent item-held-out test, broken down by benchmark")
fig.savefig(RUN_DIR / "item_test_by_benchmark.png", dpi=160, bbox_inches="tight")
plt.show()


## Split and subject-coverage receipts

The full 229,143-row cohort contains all eligible responses under the established policy. Repeated attempts remain repeated training observations, not new text embeddings. This is a single-seed full-data run; it does not isolate chunking's effect or establish repeated-seed uncertainty. Validation and outer tests stay separate throughout.


In [ ]:
split_tables = []
for path in sorted(RUN_DIR.glob("*/split.csv")):
    table = pd.read_csv(path)
    table["experiment"] = path.parent.name
    split_tables.append(table)
ledger = pd.concat(split_tables, ignore_index=True)
split_counts = ledger.groupby(["experiment", "partition", "benchmark_id"]).agg(
    responses=("row_id", "size"), items=("item_id", "nunique"), item_groups=("item_group", "nunique")
)
display(split_counts)
split_counts.to_csv(RUN_DIR / "split_counts.csv")
print("Data audit:", json.loads((RUN_DIR / "manifest.json").read_text())["data_audit"]["eligible_rows"], "eligible rows")
print("Results and plots:", RUN_DIR)
